# $\Omega$ and $\beta$ Usage

This notebook shows the minimal workflow for computing $\beta$ from local omega analysis on one smoothed disclination line.

## JupyterLab and PyVistaQt

**If you are not running this workflow in JupyterLab, you usually do not need this setup section.**

This setup section exists ONLY to handle compatibility between JupyterLab and `pyvistaqt`. `pyvistaqt` opens a native Qt window instead of rendering inline inside the notebook. In JupyterLab, the Qt event loop must be enabled before creating that window:

```python
%gui qt
```

The `QT_API` line below chooses a Qt binding before importing `pyvistaqt` or `nematics3d`:

```python
os.environ["QT_API"] = "pyqt5"
```

If you see an incompatible Qt binding error, restart the kernel and run the notebook again from the top. A Qt binding cannot be switched cleanly after it has already been imported in the same kernel.

This also requires a local desktop session with a working Qt backend. It will not work in a headless browser-only server unless an appropriate display is available.

In [3]:
import os
os.environ["QT_API"] = "pyqt5"
%gui qt

## Imports

In [4]:
from pathlib import Path
import sys

import numpy as np


REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "nematics3d").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError("Could not locate the repository root.")
    REPO_ROOT = REPO_ROOT.parent

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import nematics3d

## Load Q Data

This notebook starts from a saved Q-tensor field as the example file `Q_1630.npy`, from Vincent's simulation.

After loading the file, the variable `Q_data` should be a 3D lattice of Q tensors. In this example, the first index of the saved array selects one frame, and the crop selects a smaller spatial region around one defect line.

`nematics3d.QFieldObject` is the main entry point for this repository. When it is created from `Q_data`, it detects disclination points and classifies them into line objects. For this example, the crop is chosen so that exactly one line is present.

In [5]:
DATA_PATH = REPO_ROOT / "tests" / "disclination" / "beta" / "Q_1630.npy"
Q_data = np.load(DATA_PATH)[0]
Q_data = Q_data[168:185, 5:32, 10:35]

Q = nematics3d.QFieldObject(
    Q=Q_data,
    name="WT",
)

[PROGRESS]
    <QFieldObject.__init__> 
    Start to initialize Q tensor `WT`.
[PROGRESS]
    <QFieldObject.__init__> 
    Start defect analysis as detecting defects and classifying them into distinct lines for Q tensor `WT` 
    This operation might take a while.
    You can disable this automatic operation by setting is_detect_defects=False and is_classify_lines=False when initializing the Q tensor.
[INFO]
        <QFieldObject[name='WT'].act_defect_detect> 
        70 defects are found.
[INFO]
        <QFieldObject[name='WT'].act_lines_classify> 
        1 lines are found.
[PROGRESS]
    <QFieldObject.__init__> 
    Defect analysis is finished, with 0.06 s


## Smooth the Single Line

Computing $\beta$ requires the local tangent direction of the disclination line at the chosen position. Raw detected defect points are discrete and may be noisy, so the line must be smoothed before `act_calc_omega(...)` can evaluate a stable tangent.

The smoothing routine has several parameters, but the most important one for basic use is `window_length`. This is NOT a physical arc length. It is the number of discrete line samples included in the local smoothing window along the ordered defect-line points. For example, `window_length=28` means the local fit/filter uses a neighborhood of 28 sampled points along the line. A larger window suppresses more small-scale noise, but it can also wash out real curvature.

A later Q&A section will discuss how to visually judge whether the selected smoothing window is appropriate for your data.

In [8]:
if len(Q.lines) != 1:
    raise RuntimeError(f"Expected exactly one line, got {len(Q.lines)}.")

Q.act_lines_smooth(window_length=28)
smooth = Q.lines[0].smooths[-1]
smooth

[INFO]
    <QFieldObject[name='WT'].act_lines_smooth> 
    No input value provided for minimum smoothed line length. 
    Using the default value self.default_miminum_line_length_smooth=61.
[INFO]
    <QFieldObject[name='WT'].act_lines_smooth> 
    There are 1 disclination lines in total, with 1 lines are smoothed.
    The smoothing window length is: 28


DisclinationLineSmooth('disclination line 0 smooth_version 2')

In this cell, `Q.act_lines_smooth(...)` is the method that creates smoothed versions of the disclination lines stored in the `Q` object. It does not smooth only `Q.lines[0]`; it goes through all classified lines in `Q.lines` and smooths the ones that satisfy the length and option checks.

`Q.lines` is the collection of all disclination-line objects detected in this `QFieldObject`. Each line object stores its own smoothed versions in `line.smooths`. This is useful because you can create multiple smoothed versions of the same raw line using different smoothing parameters.

In this example, there is only one line, so we use `Q.lines[0]`. After smoothing, `Q.lines[0].smooths[-1]` selects the most recently created smoothed version. This smoothed line object is the object used below for tangent-based omega and $\beta$ calculations.

## Compute $\beta$ at One Position

`u_percent` is the internal spline parameter of the smoothed line, expressed as a percentage from 0 to 100. The smoothed line is built from the ordered defect-line samples (the defect points detected by winding number); during smoothing, those samples are assigned a normalized parameter `u` that runs from the beginning of the ordered line to the end. `u_percent` is simply `100 * u`. It tells the code where to evaluate the smoothed spline in this normalized parameter domain.

This means `u_percent=0` is the start of the current smoothed-line parameterization, `u_percent=50` is halfway through that parameter domain, and `u_percent=100` is the end. It is not a physical arc length and it is not measured in simulation length units. The local position and tangent used for $\beta$ are evaluated from this smoothed-line spline parameter.

In [9]:
u_percent = 5
omega_result = smooth.act_calc_omega(u_percent)
beta = omega_result["beta"]
beta

83.48768772428835

For the selected `u_percent`, the computed $\beta$ value suggests a wedge-like defect. Let's visually check whether the local director pattern agrees with that interpretation.

The next cell draws the detected disclination line and then plots the director field around the same `u_percent` position.

In [11]:
Q.act_visualize_disclination_lines(
    min_line_length=0,
    title="WT disclination lines",
)

Q.act_visualize_n_near_defect(u_percent=u_percent)

[WARNING]
                <QFieldObject[name='WT'] -> _helper_check_name> 
                >>> 'WT disclination lines' already exists in Registry 'figures'! Renamed to 'WT disclination lines_1'.
                Current warning call: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\registry_base.py:111
                Caller: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\registry_base.py:155
                code: name = self._helper_check_name(term.name)
[INFO]
    <QFieldObject[name='WT'].act_visualize_disclination_lines> 
    No minimum line length has been provided for the plotted lines. Use the default value 75
[WARNING]
        <RegistryBase[name='Planes']._helper_check_name> 
        >>> 'defect section grid' already exists in Registry 'Planes' (registry of cross-section grids for the smoothed disclination line 'disclination line 0 smooth_version 2')! Renamed to 'defect section grid_1'.
        Current warning call: D:\Document\GitHub\Nematics3D\src\nematics3d\classes\r

## Create a $\beta$ Line Function

The line function samples $\beta$ along the smoothed line and registers it on `smooth.linefuncs` under the name `"beta"`.

In [7]:
beta_func = smooth.act_create_linefunc(
    func=lambda u: smooth.act_calc_omega(u)["beta"],
    u_samples=np.arange(0, 100, 5),
    name="beta",
)

smooth.linefuncs["beta"]

SmoothedLineFunc('beta'), num_samples=20, mode='wrap'

## Visualize the Smoothed Line Colored by $\beta$

This cell creates a PyVistaQt window and then switches the plotted smooth tube to scalar coloring with `beta_func`.

In [ ]:
Q.act_visualize_disclination_lines(
    min_line_length=0,
    title="WT disclination lines",
)

Q.act_visualize_n_near_defect(u_percent=u_percent)

smooth.visual.wrapped.act_commit(
    paint_by="scalars",
    resolver_source="u_percent",
    scalars=beta_func,
    scalars_cmap="viridis",
    scalar_bar_title="beta",
)

Q.figs.active_fig